# SynthKit — Interactive Quality Validation

This notebook lets you **interactively inspect and compare the quality** of synthetic text generated by SynthKit.

It uses `LocalQualityScorer` and `BenchmarkRunner` from SynthKit v1 — all scoring runs **fully locally** with no API calls.

## Contents
1. [Setup](#1-setup)
2. [Score a single text pair](#2-score-a-single-text-pair)
3. [Batch scoring with visualisation](#3-batch-scoring-with-visualisation)
4. [Score augmented outputs end-to-end](#4-score-augmented-outputs-end-to-end)
5. [Multi-provider benchmark](#5-multi-provider-benchmark)
6. [Export results](#6-export-results)

---
**Requirements:** `pip install 'synthkit[quality]'`  
For the benchmark section: `pip install 'synthkit[benchmark]'`

## 1 Setup

In [ ]:
# Install synthkit with quality extras if not already installed
# Uncomment the line below if running for the first time:
# !pip install -q 'synthkit[quality]'

import math
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

from synthkit.quality.scorer import LocalQualityScorer
from synthkit.providers.mock import MockProvider
from synthkit.config import SynthKitConfig
from synthkit.pipelines.text_augmentation import augment_texts

warnings.filterwarnings('ignore')  # suppress verbose model loading messages
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

print('SynthKit quality validation notebook ready.')

## 2 Score a single text pair

Paste any **original** and **generated** text pair to inspect all five quality metrics:

| Metric | Range | Better |
|---|---|---|
| `semantic_similarity` | −1 → 1 | higher |
| `perplexity` | > 0 | lower |
| `ngram_diversity` | 0 → 1 | higher |
| `fluency` | 0 → 1 | higher |
| `composite` | 0 → 1 | higher |

In [ ]:
# ── Edit these two strings ──────────────────────────────────────────────────
ORIGINAL  = "The quick brown fox jumps over the lazy dog."
GENERATED = "A swift auburn fox leaps across the sleepy hound."
# ────────────────────────────────────────────────────────────────────────────

scorer = LocalQualityScorer()
scores = scorer.score(ORIGINAL, GENERATED)

print('Scores:')
for k, v in scores.items():
    print(f'  {k:<25s} {v}')

In [ ]:
def plot_single_scores(scores: dict, title: str = 'Quality metrics') -> None:
    """Bar chart for a single (original, generated) pair."""
    keys   = ['semantic_similarity', 'ngram_diversity', 'fluency', 'composite']
    labels = ['Semantic\nSimilarity', 'N-gram\nDiversity', 'Fluency', 'Composite']
    vals   = [scores[k] for k in keys]
    colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']

    fig, ax = plt.subplots(figsize=(7, 3.5))
    bars = ax.bar(labels, vals, color=colors, edgecolor='white', width=0.5)
    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Score  (0–1)')
    ax.set_title(f"{title}   |   perplexity = {scores['perplexity']:.1f}")
    ax.axhline(0.5, color='grey', lw=0.8, ls='--', alpha=0.6)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9,
        )
    plt.tight_layout()
    plt.show()

plot_single_scores(scores)

## 3 Batch scoring with visualisation

Score a list of pairs at once and compare them visually.

In [ ]:
# ── Edit this list of (original, generated) pairs ───────────────────────────
PAIRS = [
    (
        "The quick brown fox jumps over the lazy dog.",
        "A swift auburn fox leaps across the sleepy hound.",
    ),
    (
        "Synthetic data helps bootstrap machine learning projects.",
        "Artificially generated data assists in kick-starting ML endeavours.",
    ),
    (
        "Climate change poses significant risks to global ecosystems.",
        "Global warming presents major threats to the world's ecological systems.",
    ),
    (
        "The neural network achieved state-of-the-art performance on benchmarks.",
        "pizza pizza pizza pizza pizza pizza pizza pizza pizza",  # bad example
    ),
]
# ────────────────────────────────────────────────────────────────────────────

batch_results = []
for i, (orig, gen) in enumerate(PAIRS):
    s = scorer.score(orig, gen)
    s['pair'] = i + 1
    s['preview'] = gen[:55] + ('…' if len(gen) > 55 else '')
    batch_results.append(s)
    print(f"Pair {i+1}: composite={s['composite']:.3f}  perplexity={s['perplexity']:.1f}")

In [ ]:
def plot_batch(results: list[dict]) -> None:
    pair_labels = [f"Pair {r['pair']}" for r in results]
    metric_keys    = ['semantic_similarity', 'ngram_diversity', 'fluency', 'composite']
    metric_display = ['Semantic sim.', 'N-gram div.', 'Fluency', 'Composite']
    colors = ['#4c72b0', '#55a868', '#c44e52', '#8172b2']

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: grouped bar chart — all 4 metrics per pair
    n_pairs   = len(results)
    n_metrics = len(metric_keys)
    width     = 0.15
    x         = np.arange(n_pairs)
    for i, (key, label, color) in enumerate(zip(metric_keys, metric_display, colors)):
        offset = (i - n_metrics / 2 + 0.5) * width
        vals = [r[key] for r in results]
        axes[0].bar(x + offset, vals, width=width, label=label, color=color, edgecolor='white')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(pair_labels)
    axes[0].set_ylim(0, 1.2)
    axes[0].set_ylabel('Score  (0–1)')
    axes[0].set_title('All metrics by pair')
    axes[0].legend(fontsize=8, loc='upper right')
    axes[0].axhline(0.5, color='grey', lw=0.8, ls='--', alpha=0.5)

    # Right: perplexity
    ppl_vals = [r['perplexity'] for r in results]
    bar_colors = [('#e64c4c' if v > 200 else '#4c72b0') for v in ppl_vals]
    axes[1].bar(pair_labels, ppl_vals, color=bar_colors, edgecolor='white')
    axes[1].set_ylabel('Perplexity  (lower = more fluent)')
    axes[1].set_title('Perplexity by pair')
    for xi, val in enumerate(ppl_vals):
        axes[1].text(xi, val + 1, f'{val:.0f}', ha='center', fontsize=8)

    plt.tight_layout()
    plt.show()

plot_batch(batch_results)

In [ ]:
# Heatmap — quick overview across all pairs and metrics
def plot_heatmap(results: list[dict]) -> None:
    metric_keys    = ['semantic_similarity', 'ngram_diversity', 'fluency', 'composite']
    metric_display = ['Semantic\nsim.', 'N-gram\ndiv.', 'Fluency', 'Composite']
    pair_labels    = [f"Pair {r['pair']}" for r in results]

    data = np.array([[r[k] for k in metric_keys] for r in results])  # shape (n_pairs, n_metrics)

    fig, ax = plt.subplots(figsize=(7, 0.7 * len(results) + 1.5))
    im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(metric_keys)))
    ax.set_xticklabels(metric_display, fontsize=9)
    ax.set_yticks(range(len(results)))
    ax.set_yticklabels(pair_labels)
    for i in range(len(results)):
        for j in range(len(metric_keys)):
            ax.text(j, i, f'{data[i, j]:.2f}', ha='center', va='center', fontsize=8,
                    color='black' if 0.3 < data[i, j] < 0.7 else 'white')
    plt.colorbar(im, ax=ax, fraction=0.04)
    ax.set_title('Quality heatmap  (green = good)')
    plt.tight_layout()
    plt.show()

plot_heatmap(batch_results)

## 4 Score augmented outputs end-to-end

Run `augment_texts` on a set of source texts and immediately score the outputs — the full v1 loop.

> **Note:** By default this uses `MockProvider` so it works without any API key or model download.  
> Swap it for `HuggingFaceProvider`, `AnthropicProvider`, or `OpenAIProvider` as needed.

In [ ]:
# ── Configure ────────────────────────────────────────────────────────────────
SOURCE_TEXTS = [
    "Synthetic data helps bootstrap machine learning projects.",
    "Data augmentation improves model robustness and generalisation.",
    "Large language models can generate diverse training examples.",
    "Quality evaluation is essential before using generated data.",
]
OPERATION = "paraphrase"

provider = MockProvider()          # replace with AnthropicProvider(...) etc.
config   = SynthKitConfig(seed=42, max_new_tokens=64, temperature=0.7)
# ────────────────────────────────────────────────────────────────────────────

records = augment_texts(SOURCE_TEXTS, operation=OPERATION, provider=provider, config=config)
print(f'Generated {len(records)} records.')
for r in records:
    print(f"  [{r['id']}] {r['augmented_text'][:80]}")

In [ ]:
# Score every record
e2e_scores = []
for r in records:
    s = scorer.score(r['source_text'], r['augmented_text'])
    s['id'] = r['id']
    s['source_text']    = r['source_text']
    s['augmented_text'] = r['augmented_text']
    e2e_scores.append(s)

# Summary statistics
metric_keys = ['semantic_similarity', 'ngram_diversity', 'fluency', 'composite', 'perplexity']
print('\nSummary statistics:')
print(f'{"Metric":<28} {"Mean":>8}  {"Min":>8}  {"Max":>8}')
print('-' * 56)
for key in metric_keys:
    vals = [s[key] for s in e2e_scores]
    print(f'{key:<28} {np.mean(vals):>8.4f}  {np.min(vals):>8.4f}  {np.max(vals):>8.4f}')

In [ ]:
# Flag low-quality outputs for review
COMPOSITE_THRESHOLD = 0.35  # adjust as needed

flagged = [s for s in e2e_scores if s['composite'] < COMPOSITE_THRESHOLD]
if flagged:
    print(f'Flagged {len(flagged)} record(s) with composite < {COMPOSITE_THRESHOLD}:')
    for s in flagged:
        print(f"  id={s['id']}  composite={s['composite']:.3f}")
        print(f"    source:    {s['source_text'][:80]}")
        print(f"    generated: {s['augmented_text'][:80]}")
else:
    print(f'All records passed the composite threshold of {COMPOSITE_THRESHOLD}.')

In [ ]:
# Radar (spider) chart — mean metric profile
def plot_radar(scores_list: list[dict], title: str = 'Mean quality profile') -> None:
    keys   = ['semantic_similarity', 'ngram_diversity', 'fluency', 'composite']
    labels = ['Semantic\nSim.', 'N-gram\nDiv.', 'Fluency', 'Composite']
    vals   = [float(np.mean([s[k] for s in scores_list])) for k in keys]

    angles = np.linspace(0, 2 * np.pi, len(keys), endpoint=False).tolist()
    vals  += vals[:1]
    angles += angles[:1]

    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'polar': True})
    ax.plot(angles, vals, 'o-', linewidth=2, color='#4c72b0')
    ax.fill(angles, vals, alpha=0.25, color='#4c72b0')
    ax.set_thetagrids(np.degrees(angles[:-1]), labels)
    ax.set_ylim(0, 1)
    ax.set_title(title, pad=15)
    plt.tight_layout()
    plt.show()

plot_radar(e2e_scores, title=f'Mean quality profile — {OPERATION} with {type(provider).__name__}')

## 5 Multi-provider benchmark

Use `BenchmarkRunner` to run the same texts through multiple providers and compare scores side-by-side.

> By default two `MockProvider` instances are used so the cell works without API keys.  
> Replace them with real providers once you have credentials.

In [ ]:
from synthkit.benchmark.runner import BenchmarkRunner

# ── Configure providers ──────────────────────────────────────────────────────
# Uncomment and fill in real providers when you have API keys:
#
# from synthkit.providers.anthropic import AnthropicProvider
# from synthkit.providers.openai    import OpenAIProvider
#
# PROVIDERS = {
#     'claude': AnthropicProvider(api_key='...'),
#     'gpt':    OpenAIProvider(api_key='...'),
# }

PROVIDERS = {
    'mock_a': MockProvider(),
    'mock_b': MockProvider(),
}

BENCHMARK_TEXTS = [
    "Synthetic data helps bootstrap machine learning projects.",
    "Data augmentation improves model robustness and generalisation.",
    "Large language models can generate diverse training examples.",
]
# ────────────────────────────────────────────────────────────────────────────

runner  = BenchmarkRunner(config=SynthKitConfig(seed=42, max_new_tokens=64))
bm_results = runner.run(providers=PROVIDERS, texts=BENCHMARK_TEXTS, operation='paraphrase')

print('Benchmark summary:')
for name, result in bm_results.items():
    print(f'\n  {name}:')
    for k, v in result.summary.items():
        print(f'    {k:<35s} {v}')

In [ ]:
def plot_benchmark(bm_results: dict, title: str = 'Provider benchmark') -> None:
    provider_names = list(bm_results.keys())
    metric_keys    = ['mean_semantic_similarity', 'mean_ngram_diversity', 'mean_fluency', 'mean_composite']
    metric_display = ['Semantic sim.', 'N-gram div.', 'Fluency', 'Composite']
    palette = ['#4c72b0', '#55a868', '#c44e52', '#8172b2', '#ccb974']

    n_metrics   = len(metric_keys)
    n_providers = len(provider_names)
    width = 0.7 / n_providers
    x    = np.arange(n_metrics)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for i, name in enumerate(provider_names):
        summary = bm_results[name].summary
        offset  = (i - n_providers / 2 + 0.5) * width
        vals    = [summary.get(k, 0.0) for k in metric_keys]
        bars    = axes[0].bar(
            x + offset, vals, width=width,
            label=name, color=palette[i % len(palette)], edgecolor='white',
        )
        for bar, val in zip(bars, vals):
            axes[0].text(
                bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=7,
            )

    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metric_display, fontsize=9)
    axes[0].set_ylim(0, 1.2)
    axes[0].set_ylabel('Mean score  (0–1)')
    axes[0].set_title(f'{title} — quality metrics')
    axes[0].legend(fontsize=9)
    axes[0].axhline(0.5, color='grey', lw=0.8, ls='--', alpha=0.5)

    ppl_vals = [bm_results[n].summary.get('mean_perplexity', 0.0) for n in provider_names]
    bar_colors = [palette[i % len(palette)] for i in range(n_providers)]
    axes[1].bar(provider_names, ppl_vals, color=bar_colors, edgecolor='white')
    axes[1].set_ylabel('Mean perplexity  (lower = more fluent)')
    axes[1].set_title(f'{title} — perplexity')
    for xi, val in enumerate(ppl_vals):
        axes[1].text(xi, val + 0.5, f'{val:.1f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.show()

plot_benchmark(bm_results)

In [ ]:
# Per-record score distribution (violin plot)
def plot_score_distributions(bm_results: dict, metric: str = 'composite') -> None:
    provider_names = list(bm_results.keys())
    data = []
    labels = []
    for name in provider_names:
        vals = [s[metric] for s in bm_results[name].scores if metric in s]
        if vals:
            data.append(vals)
            labels.append(name)

    if not data:
        print('No data to plot.')
        return

    fig, ax = plt.subplots(figsize=(7, 4))
    parts = ax.violinplot(data, showmedians=True)
    for pc in parts['bodies']:
        pc.set_facecolor('#4c72b0')
        pc.set_alpha(0.5)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels)
    ax.set_ylabel(f'{metric} score')
    ax.set_title(f'Score distribution per provider — {metric}')
    plt.tight_layout()
    plt.show()

plot_score_distributions(bm_results, metric='composite')

## 6 Export results

Save the benchmark summary and per-record scores for offline analysis.

In [ ]:
OUTPUT_DIR = Path('outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Summary per provider
summary_path = OUTPUT_DIR / 'benchmark_summary.json'
summary_export = {name: res.summary for name, res in bm_results.items()}
summary_path.write_text(json.dumps(summary_export, indent=2))
print(f'Summary written to {summary_path}')

# Per-record scores
records_path = OUTPUT_DIR / 'benchmark_records.json'
records_export = {
    name: {
        'summary': res.summary,
        'scores':  res.scores,
        'records': res.records,
    }
    for name, res in bm_results.items()
}
records_path.write_text(json.dumps(records_export, indent=2))
print(f'Records written  to {records_path}')

# End-to-end scores from section 4
e2e_path = OUTPUT_DIR / 'e2e_scores.json'
e2e_path.write_text(json.dumps(e2e_scores, indent=2))
print(f'E2E scores written to {e2e_path}')